#  Retail Sales Data Analysis 

**Goal:** Clean a raw retail orders dataset, perform exploratory data analysis (EDA), engineer features, 
and extract business insights on revenue, customers, discounts, and delivery performance.

**Dataset columns:** `order_id`, `order_date`, `customer_id`, `product_category`, `region`, `quantity`, 
`unit_price`, `discount`, `payment_method`, `delivery_days`, `customer_rating`, `revenue`


## 1. Setup & Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)


## 2. Load Data


In [ ]:
# Update the path to wherever your file lives
df = pd.read_excel('C:/Users/Tadas/Documents/Projects/Portfolios/Sales_Data_Analysis/ecommerce_sales_analytics_5000.xlsx')  # or pd.read_excel('sales_data.xlsx')

print(df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


## 3. Data Cleaning

###  Missing Values


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct.round(2)}).sort_values('missing_count', ascending=False)


In [ ]:
# Example strategy — adjust based on what you find above
# Numeric columns: fill with median
num_cols = ['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue']
for c in num_cols:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())

# Categorical columns: fill with mode
cat_cols = ['product_category', 'region', 'payment_method']
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].mode()[0])

df.isnull().sum().sum()  # should be 0 now


###  Data Types


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['order_id'] = df['order_id'].astype(str)
df['customer_id'] = df['customer_id'].astype(str)

df.dtypes


###  Duplicates


In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate order_ids:', df['order_id'].duplicated().sum())

df = df.drop_duplicates(subset='order_id', keep='first')


###  Categorical Consistency (typos, casing)


In [ ]:
for c in ['product_category', 'region', 'payment_method']:
    print(c, '->', sorted(df[c].unique()))

# Standardize casing/whitespace
for c in ['product_category', 'region', 'payment_method']:
    df[c] = df[c].astype(str).str.strip().str.title()


###  Logical / Range Checks

- `discount` should be between 0 and 1
- `quantity` and `unit_price` should be positive
- `customer_rating` should be within a valid scale (assume 1–5)
- `delivery_days` shouldn't be negative
- `revenue` should roughly equal `quantity * unit_price * (1 - discount)`


In [ ]:
issues = {}
issues['invalid_discount'] = df[(df['discount'] < 0) | (df['discount'] > 1)].shape[0]
issues['non_positive_quantity'] = df[df['quantity'] <= 0].shape[0]
issues['non_positive_price'] = df[df['unit_price'] <= 0].shape[0]
issues['rating_out_of_range'] = df[(df['customer_rating'] < 1) | (df['customer_rating'] > 5)].shape[0]
issues['negative_delivery_days'] = df[df['delivery_days'] < 0].shape[0]

issues


In [ ]:
# Recompute expected revenue and flag mismatches

df['expected_revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount'])
df['revenue_diff'] = (df['revenue'] - df['expected_revenue']).abs()

mismatch_threshold = 1.0  # allow small rounding differences
mismatches = df[df['revenue_diff'] > mismatch_threshold]
print(f'Rows with revenue mismatch: {len(mismatches)} ({len(mismatches)/len(df)*100:.2f}%)')
mismatches[['order_id', 'quantity', 'unit_price', 'discount', 'revenue', 'expected_revenue']].head()


In [ ]:
# Decide: correct revenue using the formula (common in portfolio cleanup), or drop/flag mismatches
df['revenue_corrected'] = df['expected_revenue']
df.drop(columns=['expected_revenue', 'revenue_diff'], inplace=True)


### Outlier Detection (IQR method)


In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return series[(series < lower) | (series > upper)]

for c in ['unit_price', 'quantity', 'revenue_corrected', 'delivery_days']:
    out = iqr_outliers(df[c])
    print(f'{c}: {len(out)} outliers')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, c in zip(axes, ['unit_price', 'quantity', 'revenue_corrected', 'delivery_days']):
    sns.boxplot(y=df[c], ax=ax, color='skyblue')
    ax.set_title(f'Boxplot: {c}')
plt.tight_layout()
plt.show()


> ✅ **Cleaning summary so far:** handled missing values, fixed dtypes, removed duplicates, 
standardized categories, validated logical ranges, corrected revenue mismatches, and identified outliers 
(kept unless clearly erroneous — flag your decision here in your own notebook).


## 4. Exploratory Data Analysis

###  Univariate Distributions


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
num_features = ['revenue_corrected', 'unit_price', 'quantity', 'discount', 'delivery_days', 'customer_rating']
for ax, c in zip(axes.flatten(), num_features):
    sns.histplot(df[c], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {c}')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, c in zip(axes, ['region', 'product_category', 'payment_method']):
    sns.countplot(data=df, x=c, ax=ax, palette='viridis', order=df[c].value_counts().index)
    ax.set_title(f'Order Count by {c}')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


###  Revenue by Category / Region / Payment Method


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, c in zip(axes, ['region', 'product_category', 'payment_method']):
    rev = df.groupby(c)['revenue_corrected'].sum().sort_values(ascending=False)
    sns.barplot(x=rev.index, y=rev.values, ax=ax, palette='mako')
    ax.set_title(f'Total Revenue by {c}')
    ax.set_ylabel('Revenue')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


###  Correlation Heatmap


In [ ]:
plt.figure(figsize=(8, 6))
corr_cols = ['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue_corrected']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


###  Discount vs Revenue / Quantity


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='discount', y='quantity', ax=axes[0], alpha=0.5)
axes[0].set_title('Discount vs Quantity Sold')
sns.scatterplot(data=df, x='discount', y='revenue_corrected', ax=axes[1], alpha=0.5, color='darkorange')
axes[1].set_title('Discount vs Revenue')
plt.tight_layout()
plt.show()


###  Delivery Days vs Customer Rating


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='delivery_days', y='customer_rating', color='lightgreen')
plt.title('Customer Rating by Delivery Days')
plt.xticks(rotation=45)
plt.show()


###  Revenue Trend Over Time


In [ ]:
monthly_rev = df.groupby('order_month')['revenue_corrected'].sum().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_rev, x='order_month', y='revenue_corrected', marker='o')
plt.title('Monthly Revenue Trend')
plt.xticks(rotation=45)
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()


In [ ]:
weekday_rev = df.groupby('order_weekday')['revenue_corrected'].sum()
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_rev = weekday_rev.reindex(weekday_order)

plt.figure(figsize=(10, 5))
sns.barplot(x=weekday_rev.index, y=weekday_rev.values, palette='crest')
plt.title('Revenue by Day of Week')
plt.ylabel('Revenue')
plt.show()


###  Customer-Level Analysis (RFM-style)


In [ ]:
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg(
    recency=('order_date', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('revenue_corrected', 'sum')
).reset_index()

rfm.sort_values('monetary', ascending=False).head(10)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(rfm['recency'], ax=axes[0], kde=True, color='tomato')
axes[0].set_title('Recency Distribution')
sns.histplot(rfm['frequency'], ax=axes[1], kde=True, color='seagreen')
axes[1].set_title('Frequency Distribution')
sns.histplot(rfm['monetary'], ax=axes[2], kde=True, color='steelblue')
axes[2].set_title('Monetary Distribution')
plt.tight_layout()
plt.show()


###  Top Region × Category Revenue Combinations


In [ ]:
pivot = df.pivot_table(index='region', columns='product_category', values='revenue_corrected', aggfunc='sum', fill_value=0)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Revenue by Region and Product Category')
plt.show()


## 6. Key Insights 
- **Top revenue driver:** Region/category combination generating the most revenue.
- **Discount effectiveness:** Does higher discount correlate with more quantity sold, or mostly erode margin?
- **Delivery vs satisfaction:** Does longer delivery time reduce average customer rating?
- **Customer value:** Top 10% of customers contribute what % of total revenue?
- **Seasonality:** Which month/weekday sees peak revenue?

> Replace these bullets with data-driven statements once you've run the notebook on your real dataset — 
this is what makes a portfolio project compelling to reviewers.
